# 02 CEEMDAN decomposition

Full-sample CEEMDAN is used here to inspect component structure and boundary behavior. This is the leaky full-sample decomposition later labelled A.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.volatility import load_rv_series
from src.decomposition import decompose_ceemdan

FIG_DIR = REPO_ROOT / "results" / "figures"
TABLE_DIR = REPO_ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["axes.unicode_minus"] = False

## Load and decompose the full series

In [ ]:
rv, source = load_rv_series(REPO_ROOT / "data" / "rv_series.csv", allow_network=False)
imfs = decompose_ceemdan(rv.values, trials=100, parallel=False, seed=42)
print("source:", source)
print("IMF matrix:", imfs.shape)
print("component totals:", np.round(np.sum(imfs, axis=1), 10))

## Plot all components

In [ ]:
n_imfs = imfs.shape[0]
fig, axes = plt.subplots(n_imfs + 1, 1, figsize=(14, 2.0 * (n_imfs + 1)), sharex=True)
axes[0].plot(rv.index, rv.values, color="black", linewidth=1.0)
axes[0].set_ylabel("RV")
axes[0].set_title("510300 Garman-Klass variance")
axes[0].grid(alpha=0.25)
for idx in range(n_imfs):
    axes[idx + 1].plot(rv.index, imfs[idx], linewidth=0.9)
    axes[idx + 1].set_ylabel(f"IMF {idx + 1}" if idx < n_imfs - 1 else "Residual")
    axes[idx + 1].grid(alpha=0.25)
axes[-1].set_xlabel("Date")
fig.tight_layout()
fig.savefig(FIG_DIR / "ceemdan_decomposition.png", dpi=200, bbox_inches="tight")
plt.show()

## Boundary-energy diagnostic

A component whose first and last 10% contain much more energy than their 20% sample share is a boundary-effect candidate.

In [ ]:
boundary = max(1, int(0.10 * len(rv)))
rows = []
for idx, component in enumerate(imfs):
    total = np.sum(component**2)
    edge = np.sum(component[:boundary]**2) + np.sum(component[-boundary:]**2)
    rows.append(
        {
            "component": f"IMF{idx + 1}" if idx < n_imfs - 1 else "Residual",
            "energy": total,
            "boundary_energy_share": edge / total if total > 0 else np.nan,
        }
    )
boundary_table = pd.DataFrame(rows)
boundary_table.to_csv(TABLE_DIR / "ceemdan_boundary_energy.csv", index=False)
boundary_table